# TabTune · Enterprise Time Series for Banking
### From a raw multi-entity ledger to liquidity forecasts and anomaly alerts

**Library:** TabTune open source (`tabtune.TimeSeries`) · **Runtime:** Colab Pro+ with a GPU (A100 or L4)

| Part | What you will see |
|---|---|
| 1 | An enterprise ledger: 4 legal entities, 3 business lines, 8 products, 576 accounts, about 1 million daily postings in 4 currencies |
| 2 | **Data preparation, visualised**: duplicates, nulls, unit errors, label drift, late postings, a source-system outage, FX conversion |
| 3 | **Feature engineering, visualised**: scaling, seasonality, autocorrelation, decomposition, calendar and market covariates, feature importance |
| 4 | Model race: foundation models vs baselines on rolling backtests |
| 5 | 4-week forecast with covariates and calibrated bands |
| 6 | Group liquidity dashboard, rolled up by entity and business line |
| 7 | Ensembles and series similarity |
| 8 | Anomaly detection on a live feed with labelled incidents |

> **Data note.** The ledger is synthetic and generated here, so it can be shared freely. It is engineered to behave like a GCC banking group: entity-specific weekends and holidays, Ramadan and Eid, payroll cycles, quarter-end funding, oil- and rate-sensitive corporate flows, four currencies, and the data-quality defects a real extract carries. The companion notebook runs the same pipeline on real public cash-flow data.

**Run:** *Runtime → Change runtime type → GPU*, then *Run all*. First run downloads model weights (about 1–2 GB).

In [ ]:
# ⚙️ Install TabTune (open source) on Colab — about 2–3 minutes
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "tabtune[colab] @ git+https://github.com/Lexsi-Labs/TabTune.git", "plotly"], check=True)
print("Colab:", IN_COLAB)

In [ ]:
# 🔧 Imports and shared settings
import logging, time, warnings, re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from tabtune.logger import setup_logger
from tabtune.TimeSeries import (TimeSeriesPipeline, TimeSeriesSchema, TimeSeriesLeaderboard,
                                TimeSeriesEnsemble, split_horizon)

warnings.filterwarnings("ignore")
setup_logger(level=logging.WARNING)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.max_columns", 40)

# FAST=1 swaps foundation models for weight-free models (smoke test without GPU or downloads)
FAST = os.environ.get("TABTUNE_DEMO_FAST") == "1"
QUANTILES = [0.05, 0.1, 0.5, 0.9, 0.95]
FOUNDATION_MODELS = [] if FAST else ["ChronosBolt", "Chronos2"]     # Apache-2.0 weights: commercial-safe
BASELINES = ["SeasonalNaive", "WindowAverage", "TabularTS-GBM"]
PRIMARY = "TabularTS-GBM" if FAST else "Chronos2"                   # reads covariates
DETECTOR = "SeasonalNaive" if FAST else "ChronosBolt"
EMBEDDER = None if FAST else "ChronosBolt"
RUN_FINETUNE = False                                                # optional LoRA section

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
BLUE, ORANGE, TEAL, GREY, DARK, RED, GOLD = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D", "#E8C547"
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MON = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print(f"Device: {DEVICE} · primary model: {PRIMARY} · detector: {DETECTOR} · fast mode: {FAST}")

In [ ]:
# 🔧 Notebook settings
HORIZON, SEASON, MONTH_LAG, SCALE_WINDOW = 28, 7, 28, 365     # daily data: 4-week horizon, weekly cycle
UNIT = "USD mn"
GROUP_COLORS = {"Retail": BLUE, "Corporate": ORANGE, "Treasury": TEAL}
ENTITY_COLORS = {"KSA": BLUE, "BHR": ORANGE, "UAE": TEAL, "UK": GREY}

## 1 · The enterprise ledger

**Structure** — every row is one account's posting for one day:

`txn_id · value_date · posting_date · entity · business_line · product · account_id · currency · amount_lcy_mn · source_system`

| Entity | Currency | Weekend | Calendar |
|---|---|---|---|
| KSA | SAR (pegged 3.75) | Fri–Sat | Ramadan, Eid, 27th payroll, National Day |
| BHR | BHD (pegged 0.376) | Fri–Sat | Ramadan, Eid, 27th payroll, National Day |
| UAE | AED (pegged 3.6725) | Sat–Sun | Ramadan, Eid, 27th payroll, National Day |
| UK | GBP (floating) | Sat–Sun | Christmas, bank holidays, month-end payroll |

**Market drivers:** Brent crude (drives GCC supplier and trade-finance flows with a lag), policy rate (drives interbank funding) and GBP/USD.

In [ ]:
# 🏗️ Generate market drivers, entity calendars and series-level flows
DATES = pd.date_range("2022-01-01", "2026-10-31", freq="D")
NT = len(DATES)
rng = np.random.default_rng(7)

RAMADAN = [("2022-04-02", "2022-05-01"), ("2023-03-23", "2023-04-20"), ("2024-03-11", "2024-04-09"),
           ("2025-03-01", "2025-03-29"), ("2026-02-18", "2026-03-19")]      # approximate (moon sighting)
EID_FITR = ["2022-05-02", "2023-04-21", "2024-04-10", "2025-03-30", "2026-03-20"]
EID_ADHA = ["2022-07-09", "2023-06-28", "2024-06-16", "2025-06-06", "2026-05-27"]
ENTITIES = {"KSA": dict(ccy="SAR", weekend=(4, 5), gcc=True,  size=1.00, national=(9, 23)),
            "BHR": dict(ccy="BHD", weekend=(4, 5), gcc=True,  size=0.40, national=(12, 16)),
            "UAE": dict(ccy="AED", weekend=(5, 6), gcc=True,  size=0.60, national=(12, 2)),
            "UK":  dict(ccy="GBP", weekend=(5, 6), gcc=False, size=0.35, national=(12, 25))}
#            line,        product,                      base USD mn, noise, accounts
PRODUCTS = [("Retail",    "Branch cash withdrawals",       6.0, 0.08, 18),
            ("Retail",    "ATM withdrawals",               3.5, 0.07, 18),
            ("Retail",    "Card spend",                   12.0, 0.06, 18),
            ("Corporate", "Payroll disbursements",        40.0, 0.10, 18),
            ("Corporate", "Supplier payments",            90.0, 0.14, 18),
            ("Corporate", "Trade finance settlements",    55.0, 0.30, 18),
            ("Treasury",  "Interbank funding",           160.0, 0.12, 18),
            ("Treasury",  "FX settlements",              110.0, 0.18, 18)]

# --- market drivers -------------------------------------------------------------------------------
brent = np.empty(NT); brent[0] = 82.0
for t in range(1, NT):                                   # mean-reverting with occasional shocks
    brent[t] = brent[t - 1] + 0.01 * (80 - brent[t - 1]) + rng.normal(0, 1.3) + (rng.random() < 0.004) * rng.normal(0, 7)
brent = np.clip(brent, 45, 130)
rate_path = [("2022-01-01", 1.25), ("2022-06-01", 2.75), ("2022-12-01", 5.0), ("2023-07-01", 6.0),
             ("2024-10-01", 5.5), ("2025-03-01", 5.0), ("2025-10-01", 4.5)]
policy_rate = pd.Series(np.nan, index=DATES)
for d0, r in rate_path:
    policy_rate[pd.Timestamp(d0):] = r
gbpusd = 1.27 * np.exp(np.cumsum(rng.normal(0, 0.004, NT)) - np.linspace(0, 0.02, NT))
oil_5d_move = np.abs(pd.Series(brent).pct_change(5).fillna(0).to_numpy())
brent_lag30 = pd.Series(brent).shift(30).bfill().to_numpy()
market = pd.DataFrame({"timestamp": DATES, "brent_usd": brent, "policy_rate": policy_rate.to_numpy(), "gbpusd": gbpusd})
LCY_PER_USD = {"SAR": np.full(NT, 3.75), "BHD": np.full(NT, 0.376), "AED": np.full(NT, 3.6725), "GBP": 1 / gbpusd}


# --- entity calendars -----------------------------------------------------------------------------
def entity_calendar(entity):
    e = ENTITIES[entity]
    cal = pd.DataFrame({"timestamp": DATES})
    dow, day, month = DATES.dayofweek, DATES.day, DATES.month
    cal["is_weekend"] = np.isin(dow, e["weekend"]).astype(int)
    ram, pre, eid = (np.zeros(NT, bool) for _ in range(3))
    if e["gcc"]:
        for s, x in RAMADAN:
            s, x = pd.Timestamp(s), pd.Timestamp(x)
            ram |= (DATES >= s) & (DATES <= x); pre |= (DATES > x - pd.Timedelta(days=7)) & (DATES <= x)
        for d0 in EID_FITR:
            d0 = pd.Timestamp(d0); eid |= (DATES >= d0) & (DATES < d0 + pd.Timedelta(days=4))
        for d0 in EID_ADHA:
            d0 = pd.Timestamp(d0); eid |= (DATES >= d0 - pd.Timedelta(days=1)) & (DATES < d0 + pd.Timedelta(days=4))
        payday = (day >= 25) & (day <= 27)
        holiday = eid | ((month == e["national"][0]) & (day == e["national"][1]))
    else:
        pre = (month == 12) & (day >= 15) & (day <= 24)                      # pre-Christmas peak
        eid = ((month == 12) & (day >= 25) & (day <= 26)) | ((month == 1) & (day == 1))
        payday = day >= (DATES.days_in_month - 2)
        holiday = eid
    cal["is_payday"] = np.asarray(payday).astype(int)
    cal["is_month_end"] = np.asarray(day >= DATES.days_in_month - 1).astype(int)
    cal["is_quarter_end"] = np.asarray(np.isin(month, [3, 6, 9, 12]) & (day >= DATES.days_in_month - 2)).astype(int)
    cal["is_ramadan"], cal["is_pre_festival"], cal["is_festival"] = ram.astype(int), np.asarray(pre).astype(int), np.asarray(eid).astype(int)
    cal["is_holiday"] = np.asarray(holiday).astype(int)
    return cal


CALENDARS = {e: entity_calendar(e) for e in ENTITIES}
t_years = np.arange(NT) / 365.25


def series_level(entity, product, base):
    """Expected daily flow (USD mn) for one entity × product, before noise."""
    c, e = CALENDARS[entity], ENTITIES[entity]
    wk, pay, me, qe = c.is_weekend.to_numpy(), c.is_payday.to_numpy(), c.is_month_end.to_numpy(), c.is_quarter_end.to_numpy()
    ram, pre, fest = c.is_ramadan.to_numpy(), c.is_pre_festival.to_numpy(), c.is_festival.to_numpy()
    pre_weekend = np.roll(wk, -1) * (1 - wk)                              # the day before the weekend
    oil = (brent_lag30 / 80) if e["gcc"] else np.ones(NT)
    b = base * e["size"]
    if product == "Branch cash withdrawals":
        return b * 1.04 ** t_years * (1 - 0.35 * wk) * (1 + 0.18 * pre_weekend) * (1 + 0.45 * pay) * (1 + 0.2 * ram) * (1 + 0.55 * pre) * (1 - 0.55 * fest)
    if product == "ATM withdrawals":
        return b * 1.05 ** t_years * (1 - 0.10 * wk) * (1 + 0.12 * pre_weekend) * (1 + 0.35 * pay) * (1 + 0.15 * ram) * (1 + 0.45 * pre) * (1 - 0.25 * fest)
    if product == "Card spend":
        return b * 1.15 ** t_years * (1 + 0.14 * wk) * (1 + 0.25 * pay) * (1 + 0.18 * ram) * (1 + 0.6 * pre) * (1 - 0.2 * fest)
    if product == "Payroll disbursements":
        return b * 1.06 ** t_years * (0.06 + 5.5 * pay) * (1 - 0.9 * wk * (1 - pay))
    if product == "Supplier payments":
        return b * 1.07 ** t_years * (1 - 0.93 * wk) * (1 + 0.6 * me) * (1 + 0.35 * qe) * (1 - 0.12 * ram) * (1 - 0.85 * fest) * oil ** 0.5
    if product == "Trade finance settlements":
        return b * 1.05 ** t_years * (1 - 0.95 * wk) * (1 + 0.25 * me) * (1 - 0.15 * ram) * (1 - 0.9 * fest) * oil ** 0.9
    if product == "Interbank funding":
        return b * 1.03 ** t_years * (1 - 0.97 * wk) * (1 + 0.45 * me) * (1 + 0.9 * qe) * (1 + 0.07 * (policy_rate.to_numpy() - 4)) * (1 - 0.8 * fest)
    if product == "FX settlements":
        return b * 1.06 ** t_years * (1 - 0.97 * wk) * (1 + 6.0 * oil_5d_move) * (1 + 0.2 * me) * (1 - 0.85 * fest)
    raise ValueError(product)


SERIES = [(e, line, prod, base, noise, n_acc) for e in ENTITIES for line, prod, base, noise, n_acc in PRODUCTS]
print(f"{len(SERIES)} series = {len(ENTITIES)} entities × {len(PRODUCTS)} products · {NT:,} days")

In [ ]:
# 🧾 Explode the series into an account-level ledger, then inject realistic defects
A = PRODUCTS[0][4]
blocks, truth_rows = [], []
for s_idx, (entity, line, product, base, noise, n_acc) in enumerate(SERIES):
    level = series_level(entity, product, base) * rng.lognormal(0, noise, NT)
    weights = rng.dirichlet(np.full(n_acc, 0.7))
    acct = level[None, :] * weights[:, None] * rng.lognormal(0, 0.25, (n_acc, NT))           # USD mn, account × day
    lcy = acct * LCY_PER_USD[ENTITIES[entity]["ccy"]][None, :]
    blocks.append(pd.DataFrame({
        "value_date": np.tile(DATES.values, n_acc),
        "entity": entity, "business_line": line, "product": product,
        "account_id": np.repeat([f"{entity}-{s_idx % len(PRODUCTS):02d}-{a:03d}" for a in range(n_acc)], NT),
        "currency": ENTITIES[entity]["ccy"], "amount_lcy_mn": lcy.ravel(),
        "source_system": f"{'CORE' if line == 'Retail' else 'GTB' if line == 'Corporate' else 'TRSY'}-{entity}"}))
    truth_rows.append(pd.DataFrame({"item_id": f"{entity} · {product}", "timestamp": DATES, "truth_usd_mn": acct.sum(axis=0)}))
ledger = pd.concat(blocks, ignore_index=True)
truth = pd.concat(truth_rows, ignore_index=True)
ledger.insert(0, "txn_id", np.arange(len(ledger)))
ledger["posting_date"] = ledger["value_date"]

q = np.random.default_rng(99)
n = len(ledger)
# 1 · late postings: booked 1–3 days after the value date
late = q.choice(n, int(n * 0.03), replace=False)
ledger.loc[late, "posting_date"] = ledger.loc[late, "value_date"] + pd.to_timedelta(q.integers(1, 4, len(late)), unit="D")
# 2 · unit errors: amount keyed in thousands instead of millions
unit_err = q.choice(n, 150, replace=False)
ledger.loc[unit_err, "amount_lcy_mn"] *= 1000
# 3 · missing amounts
nulls = q.choice(n, int(n * 0.0015), replace=False)
ledger.loc[nulls, "amount_lcy_mn"] = np.nan
# 4 · label drift: the same product spelled differently by different systems
drift = q.choice(n, int(n * 0.02), replace=False)
variants = [str.upper, lambda s: s.lower().replace(" ", "_"), lambda s: s + " ", lambda s: s.replace("settlements", "sttl")]
ledger["product"] = ledger["product"].astype(object)
ledger.loc[drift, "product"] = [variants[i % 4](p) for i, p in enumerate(ledger.loc[drift, "product"])]
# 5 · source-system outage: CORE-BHR delivered nothing for four days
OUTAGE = pd.date_range("2026-03-08", periods=4)
ledger = ledger[~((ledger.source_system == "CORE-BHR") & ledger.value_date.isin(OUTAGE))]
# 6 · replayed batches: duplicate rows
dups = ledger.sample(int(len(ledger) * 0.004), random_state=3)
ledger = pd.concat([ledger, dups], ignore_index=True).sample(frac=1, random_state=1).reset_index(drop=True)

print(f"raw ledger: {len(ledger):,} rows · {ledger.account_id.nunique()} accounts · "
      f"{ledger.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory")
display(ledger.head(6))

## 2 · Data preparation, visualised

A forecasting model is only as good as the series it is given. Each step below fixes one defect, and each plot shows what the defect looked like and what the fix did.

In [ ]:
# 🔎 Profile of the raw extract
norm = lambda p: re.sub(r"[^a-z]", "", p.lower()).replace("sttl", "settlements")     # one key per product, however it is spelled
canon = {norm(p): p for _, p, *_ in PRODUCTS}
profile = pd.DataFrame({
    "rows": [len(ledger)], "accounts": [ledger.account_id.nunique()], "source systems": [ledger.source_system.nunique()],
    "distinct product labels": [ledger["product"].nunique()], "null amounts": [int(ledger.amount_lcy_mn.isna().sum())],
    "duplicate txn_id": [int(ledger.txn_id.duplicated().sum())],
    "late postings": [int((ledger.posting_date > ledger.value_date).sum())]}).T.rename(columns={0: "raw extract"})
display(profile.style.format("{:,.0f}"))

daily_rows = ledger.groupby(["value_date", "source_system"]).size().unstack(fill_value=0)
labels = ledger["product"].value_counts()
fig = make_subplots(rows=1, cols=3, column_widths=[0.4, 0.3, 0.3],
                    subplot_titles=("Rows delivered per day by source system (2026)", "Product labels found (log count)",
                                    "Amount distribution by currency (log10)"))
recent = daily_rows[daily_rows.index >= "2026-01-01"]
fig.add_trace(go.Heatmap(z=recent.T.values, x=recent.index, y=recent.columns, colorscale=[[0, RED], [0.01, "#DCE4FB"], [1, BLUE]],
                         showscale=False), 1, 1)
fig.add_trace(go.Bar(y=[repr(l) for l in labels.index[::-1]], x=labels.values[::-1], orientation="h",
                     marker_color=[GREY if l in canon.values() else RED for l in labels.index[::-1]], showlegend=False), 1, 2)
for ccy, g in ledger.sample(60000, random_state=0).groupby("currency"):
    fig.add_trace(go.Histogram(x=np.log10(g.amount_lcy_mn.dropna().clip(lower=1e-6)), name=ccy, opacity=0.6, nbinsx=60), 1, 3)
fig.update_xaxes(type="log", row=1, col=2)
fig.update_layout(template="plotly_white", height=560, barmode="overlay",
                  title="Raw extract · red = outage (left), non-standard labels (middle); the right tail holds unit errors")
fig.show()

In [ ]:
# 🧼 Cleaning pipeline: each step is counted for the waterfall below
steps = [("raw rows", len(ledger))]
clean = ledger.drop_duplicates("txn_id")
steps.append(("− replayed duplicates", len(clean)))
clean = clean[clean.amount_lcy_mn.notna()]
steps.append(("− null amounts", len(clean)))

# label drift → canonical product names
clean["product"] = clean["product"].map(lambda p: canon[norm(p)])
n_relabelled = int((ledger["product"].map(lambda p: p not in canon.values())).sum())

# unit errors: an amount far above the account's own typical level is a thousands-for-millions slip
med = clean.groupby("account_id")["amount_lcy_mn"].transform("median")
ratio = clean.amount_lcy_mn / med.clip(lower=1e-9)
is_unit_error = ratio > 150
before_fix = clean.loc[is_unit_error, ["account_id", "value_date", "amount_lcy_mn"]].assign(ratio=ratio[is_unit_error])
clean.loc[is_unit_error, "amount_lcy_mn"] /= 1000

# FX: convert every posting to USD at the day's rate
fx = pd.DataFrame({"value_date": np.tile(DATES.values, 4), "currency": np.repeat(list(LCY_PER_USD), NT),
                   "lcy_per_usd": np.concatenate(list(LCY_PER_USD.values()))})
clean = clean.merge(fx, on=["value_date", "currency"])
clean["amount_usd_mn"] = clean.amount_lcy_mn / clean.lcy_per_usd
clean["item_id"] = clean.entity + " · " + clean["product"]

fig = make_subplots(rows=1, cols=3, column_widths=[0.34, 0.33, 0.33],
                    subplot_titles=("Rows through the pipeline", "Unit errors: amount ÷ account median", "Late postings: days after value date"))
fig.add_trace(go.Waterfall(x=[s for s, _ in steps], measure=["absolute", "relative", "relative"],
                           y=[steps[0][1], steps[1][1] - steps[0][1], steps[2][1] - steps[1][1]],
                           text=[f"{steps[0][1]:,}", f"{steps[1][1] - steps[0][1]:,}", f"{steps[2][1] - steps[1][1]:,}"],
                           decreasing=dict(marker_color=RED), totals=dict(marker_color=BLUE), showlegend=False), 1, 1)
samp = ratio.sample(40000, random_state=0)
fig.add_trace(go.Histogram(x=np.log10(pd.concat([samp, ratio[is_unit_error]]).clip(lower=1e-3)), nbinsx=80, marker_color=GREY, showlegend=False), 1, 2)
fig.add_vline(x=np.log10(150), line_dash="dot", line_color=RED, row=1, col=2)
lag_days = (clean.posting_date - clean.value_date).dt.days
fig.add_trace(go.Bar(x=["on time", "1 day", "2 days", "3 days"], y=[int((lag_days == k).sum()) for k in range(4)],
                     marker_color=[GREY, ORANGE, ORANGE, ORANGE], showlegend=False), 1, 3)
fig.update_yaxes(type="log", row=1, col=2); fig.update_yaxes(type="log", row=1, col=3)
fig.update_yaxes(range=[steps[2][1] * 0.99, steps[0][1] * 1.002], row=1, col=1)
fig.update_xaxes(title_text="log10 ratio (red line = rule)", row=1, col=2)
fig.update_layout(template="plotly_white", height=460, title="Cleaning the ledger")
fig.show()
print(f"relabelled {n_relabelled:,} rows to {clean['product'].nunique()} canonical products · fixed {int(is_unit_error.sum())} unit errors "
      f"· aggregating on value_date keeps {int((lag_days > 0).sum()):,} late postings on the right day")

In [ ]:
# 🧮 Aggregate to one series per entity × product, find gaps, impute the outage
agg = clean.groupby(["item_id", "value_date"], observed=True)["amount_usd_mn"].sum().rename("value").reset_index()
grid = pd.MultiIndex.from_product([sorted(agg.item_id.unique()), DATES], names=["item_id", "timestamp"])
panel = agg.rename(columns={"value_date": "timestamp"}).set_index(["item_id", "timestamp"]).reindex(grid).reset_index()
gaps = panel[panel.value.isna()]
print(f"{panel.item_id.nunique()} series × {NT:,} days · missing series-days after aggregation: {len(gaps)}")
display(gaps.groupby("item_id").timestamp.agg(["min", "max", "count"]))

# impute a gap with the same weekday's median over the previous four weeks, scaled by the latest week's level
def impute(g):
    v = g["value"].copy()
    for i in np.flatnonzero(v.isna().to_numpy()):
        prior = [v.iloc[i - 7 * k] for k in range(1, 5) if i - 7 * k >= 0 and pd.notna(v.iloc[i - 7 * k])]
        v.iloc[i] = float(np.median(prior)) if prior else float(v.dropna().median())
    return v
panel["was_imputed"] = panel.value.isna()
panel["value"] = panel.groupby("item_id", group_keys=False).apply(impute)

# what a naive pipeline would have produced, for comparison with the truth
naive = (ledger.assign(item_id=ledger.entity + " · " + ledger["product"].astype(str))
         .merge(fx, on=["value_date", "currency"])
         .assign(usd=lambda d: d.amount_lcy_mn / d.lcy_per_usd)
         .groupby(["item_id", "posting_date"])["usd"].sum().rename("naive").reset_index().rename(columns={"posting_date": "timestamp"}))
check = panel.merge(truth, on=["item_id", "timestamp"]).merge(naive, on=["item_id", "timestamp"], how="left").fillna({"naive": 0})
err = lambda col: (check[col] - check.truth_usd_mn).abs().sum() / check.truth_usd_mn.sum() * 100

show = "BHR · Branch cash withdrawals"
w = check[(check.item_id == show) & (check.timestamp >= "2026-02-15") & (check.timestamp <= "2026-03-31")]
fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38],
                    subplot_titles=(f"{show} · around the outage", "Error vs ground truth, all series (% of volume)"))
fig.add_trace(go.Scatter(x=w.timestamp, y=w.naive, name="naive aggregation of the raw extract", line=dict(color=RED, dash="dot")), 1, 1)
fig.add_trace(go.Scatter(x=w.timestamp, y=w.value, name="after preparation", line=dict(color=BLUE, width=2.5)), 1, 1)
fig.add_trace(go.Scatter(x=w.timestamp, y=w.truth_usd_mn, name="ground truth", mode="markers", marker=dict(color=DARK, size=5)), 1, 1)
fig.add_vrect(x0=OUTAGE[0], x1=OUTAGE[-1], fillcolor=GOLD, opacity=0.25, line_width=0, row=1, col=1)
fig.add_trace(go.Bar(x=["naive aggregation", "after preparation"], y=[err("naive"), err("value")], marker_color=[RED, BLUE],
                     text=[f"{err('naive'):.1f}%", f"{err('value'):.2f}%"], textposition="outside", showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=460, legend=dict(orientation="h", y=-0.15),
                  title="Does the preparation recover the true flows?")
fig.show()

In [ ]:
# 🧩 Attach metadata and covariates: entity calendar (known in advance) and market drivers (observed)
meta = pd.DataFrame([{"item_id": f"{e} · {p}", "entity": e, "group": line, "product": p, "currency": ENTITIES[e]["ccy"]}
                     for e, line, p, *_ in SERIES])
cal_long = pd.concat([c.assign(entity=e) for e, c in CALENDARS.items()])
panel = (panel.merge(meta[["item_id", "entity"]], on="item_id")
         .merge(cal_long, on=["entity", "timestamp"]).merge(market[["timestamp", "brent_usd", "policy_rate"]], on="timestamp")
         .drop(columns="entity"))
KNOWN = ["is_weekend", "is_payday", "is_month_end", "is_quarter_end", "is_ramadan", "is_pre_festival", "is_festival", "is_holiday"]
PAST = ["brent_usd", "policy_rate"]
COV_DOC = {"is_weekend": "entity weekend (Fri–Sat in KSA and Bahrain, Sat–Sun in UAE and UK)",
           "is_payday": "payroll window (25th–27th in the GCC, last three days of the month in the UK)",
           "is_month_end": "last two days of the month", "is_quarter_end": "last three days of a quarter",
           "is_ramadan": "Ramadan (GCC entities)", "is_pre_festival": "week before Eid al-Fitr, or the pre-Christmas weeks in the UK",
           "is_festival": "Eid holidays, or Christmas and New Year in the UK", "is_holiday": "public holiday",
           "brent_usd": "Brent crude, USD per barrel", "policy_rate": "policy rate, %"}
DECOMPOSE_ITEM = "KSA · Branch cash withdrawals"

fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2}, None], [{}, {}]], vertical_spacing=0.14,
                    subplot_titles=("Group flows by business line (weekly, USD mn)", "Brent crude and policy rate", "Share of volume by entity and line"))
wk = panel.assign(group=panel.item_id.map(meta.set_index("item_id")["group"])).groupby(["group", pd.Grouper(key="timestamp", freq="W")])["value"].sum().reset_index()
for gname, g in wk.groupby("group"):
    fig.add_trace(go.Scatter(x=g.timestamp, y=g.value, name=gname, line=dict(color=GROUP_COLORS[gname], width=2)), 1, 1)
for s0, e0 in RAMADAN:
    fig.add_vrect(x0=s0, x1=e0, fillcolor=GOLD, opacity=0.18, line_width=0, row=1, col=1)
fig.add_trace(go.Scatter(x=market.timestamp, y=market.brent_usd, name="Brent (USD)", line=dict(color=DARK)), 2, 1)
fig.add_trace(go.Scatter(x=market.timestamp, y=market.policy_rate * 15, name="policy rate × 15", line=dict(color=RED, dash="dot")), 2, 1)
vol = panel.merge(meta, on="item_id").groupby(["entity", "group"])["value"].sum().unstack()
for gname in vol.columns:
    fig.add_trace(go.Bar(x=vol.index, y=vol[gname], marker_color=GROUP_COLORS[gname], showlegend=False), 2, 2)
fig.update_layout(template="plotly_white", height=760, barmode="stack", title="The prepared panel · shaded bands are Ramadan",
                  legend=dict(orientation="h", y=1.07))
fig.show()
print(f"model panel: {len(panel):,} rows · {panel.item_id.nunique()} series · {len(KNOWN)} calendar covariates · {len(PAST)} market drivers")

## 3 · Feature engineering, visualised

Foundation models take the raw history plus covariates, so the engineering effort goes into **what we hand them**: a comparable scale, the right calendar flags and the market drivers. The plots below justify each choice, and a quick gradient-boosting probe measures what the engineered features are worth.

In [ ]:
# 🧰 Feature-engineering helpers used by the plots below
GROUP_OF = meta.set_index("item_id")["group"]


def indexed(panel):
    """Each series divided by its own mean, so series of different size can share an axis."""
    return panel.assign(group=panel.item_id.map(GROUP_OF),
                        idx=panel.value / panel.groupby("item_id")["value"].transform("mean"))


def eta2(v, key):
    """Share of variance explained by the group means of `key` (0 = no pattern, 1 = fully periodic)."""
    m = v.groupby(key).transform("mean")
    tot = ((v - v.mean()) ** 2).sum()
    return float(1 - ((v - m) ** 2).sum() / tot) if tot > 0 else 0.0


def seasonality_strength(panel):
    rows = []
    for item, g in panel.groupby("item_id"):
        v = np.log1p(g["value"].clip(lower=0)).reset_index(drop=True)
        t = g["timestamp"].reset_index(drop=True)
        v = v - v.rolling(SEASON * 26, center=True, min_periods=SEASON * 4).mean().bfill().ffill()   # remove trend
        rows.append({"item_id": item, "day of week": eta2(v, t.dt.dayofweek), "day of month": eta2(v, t.dt.day),
                     "month of year": eta2(v, t.dt.month)})
    return pd.DataFrame(rows).set_index("item_id")


def acf(x, max_lag):
    x = np.asarray(x, float) - np.mean(x)
    d = float(np.dot(x, x)) or 1.0
    return np.array([np.dot(x[:-k], x[k:]) / d for k in range(1, max_lag + 1)])


def decompose(g):
    """Multiplicative split: trend × weekly pattern × day-of-month pattern × remainder."""
    v = g.set_index("timestamp")["value"].astype(float)
    trend = v.rolling(SEASON * 13, center=True, min_periods=SEASON * 4).mean().bfill().ffill().clip(lower=1e-9)
    r1 = v / trend
    weekly = r1.groupby(r1.index.dayofweek).transform("mean").clip(lower=1e-9)
    r2 = r1 / weekly
    monthly = r2.groupby(r2.index.day).transform("mean").clip(lower=1e-9)
    return pd.DataFrame({"observed": v, "trend": trend, "weekly pattern": weekly,
                         "day-of-month pattern": monthly, "remainder": r2 / monthly})


def engineer_features(panel):
    """Lag, rolling and calendar features per series. Every lag is shifted, so nothing looks ahead."""
    out = []
    for item, g in panel.sort_values("timestamp").groupby("item_id"):
        f = pd.DataFrame({"item_id": item, "timestamp": g["timestamp"].values, "y": g["y"].values})
        y = f["y"]
        for l in (1, SEASON, 2 * SEASON, MONTH_LAG):
            f[f"lag_{l}"] = y.shift(l)
        f[f"roll_mean_{SEASON}"] = y.shift(1).rolling(SEASON).mean()
        f[f"roll_mean_{4 * SEASON}"] = y.shift(1).rolling(4 * SEASON).mean()
        f[f"roll_std_{4 * SEASON}"] = y.shift(1).rolling(4 * SEASON).std()
        f["same_weekday_mean_4"] = sum(y.shift(SEASON * k) for k in range(1, 5)) / 4
        f["dow"] = f["timestamp"].dt.dayofweek
        f["dom"] = f["timestamp"].dt.day
        f["month"] = f["timestamp"].dt.month
        f["days_to_month_end"] = f["timestamp"].dt.days_in_month - f["timestamp"].dt.day
        for c in KNOWN:
            f[c] = g[c].values
        for c in PAST:
            f[f"{c}_lag1"] = pd.Series(g[c].values).shift(1).values      # observed only up to yesterday
        out.append(f)
    return pd.concat(out, ignore_index=True).dropna()


def feature_family(name):
    if name.startswith("lag_") or name == "same_weekday_mean_4":
        return "lags"
    if name.startswith("roll_"):
        return "rolling statistics"
    if any(name.startswith(p) for p in PAST):
        return "market drivers"
    return "calendar"


FAMILY_COLORS = {"lags": BLUE, "rolling statistics": TEAL, "calendar": ORANGE, "market drivers": RED}

In [ ]:
# 📏 Model scale: each series as an index of its own recent average
# (errors become comparable between a small and a large series; forecasts are converted back for the dashboards)
panel = panel.sort_values(["item_id", "timestamp"]).reset_index(drop=True)
_in_history = panel.groupby("item_id").cumcount(ascending=False) >= HORIZON      # scale never sees the hold-out
SCALE = (panel[_in_history].groupby("item_id")["value"]
         .apply(lambda s: s.tail(SCALE_WINDOW).abs().mean()).clip(lower=1e-9).rename("scale"))
panel["y"] = panel["value"] / panel["item_id"].map(SCALE)

sample = meta.groupby("group").head(2).item_id
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Raw level ({UNIT}, log axis)", "Indexed to own recent average"))
for item in sample:
    g = panel[panel.item_id == item].set_index("timestamp").resample("W").mean(numeric_only=True)
    c = GROUP_COLORS[GROUP_OF[item]]
    fig.add_trace(go.Scatter(x=g.index, y=g["value"].clip(lower=1e-6), name=item, line=dict(color=c, width=1.5)), 1, 1)
    fig.add_trace(go.Scatter(x=g.index, y=g["y"], line=dict(color=c, width=1.5), showlegend=False), 1, 2)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_layout(template="plotly_white", height=420, title="Scaling · why we index before modelling (weekly means)")
fig.show()

In [ ]:
# 🗓️ Seasonal profiles and how strong each pattern is, per series
d = indexed(panel)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Day of week", "Day of month", "Month of year"))
for j, (key, labels) in enumerate([(d.timestamp.dt.dayofweek, DOW), (d.timestamp.dt.day, None), (d.timestamp.dt.month, MON)], 1):
    prof = d.groupby(["group", key.rename("k")])["idx"].mean().reset_index()
    for gname, gg in prof.groupby("group"):
        fig.add_trace(go.Scatter(x=[labels[k] if labels else k for k in gg.k], y=gg.idx, name=gname, mode="lines+markers",
                                 line=dict(color=GROUP_COLORS[gname], width=2), showlegend=j == 1), 1, j)
fig.update_layout(template="plotly_white", height=400, title="Average profile by group (1.0 = the series' own mean)",
                  legend=dict(orientation="h", y=-0.15))
fig.show()

strength = seasonality_strength(panel)
order = meta.sort_values("group").item_id
fig = px.imshow(strength.loc[order].T, color_continuous_scale="Blues", zmin=0, zmax=1, aspect="auto",
                title="Seasonality strength · share of variance explained by each calendar cycle")
fig.update_layout(template="plotly_white", height=360, xaxis_tickangle=-60)
fig.show()

In [ ]:
# 🔁 Autocorrelation (which lags carry signal) and a decomposition of one series
reps = meta.groupby("group").head(1).item_id.tolist()
max_lag = SEASON * 9
fig = make_subplots(rows=1, cols=len(reps), subplot_titles=reps, shared_yaxes=True)
for j, item in enumerate(reps, 1):
    a = acf(panel.loc[panel.item_id == item, "y"], max_lag)
    lags = np.arange(1, max_lag + 1)
    fig.add_trace(go.Bar(x=lags, y=a, marker_color=[ORANGE if l % SEASON == 0 else GREY for l in lags], showlegend=False), 1, j)
fig.update_layout(template="plotly_white", height=360,
                  title=f"Autocorrelation · orange bars are multiples of the weekly cycle ({SEASON} steps)")
fig.show()

dec = decompose(panel[panel.item_id == DECOMPOSE_ITEM]).iloc[-SEASON * 60:]
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03, subplot_titles=list(dec.columns))
for i, c in enumerate(dec.columns, 1):
    fig.add_trace(go.Scatter(x=dec.index, y=dec[c], line=dict(color=[DARK, BLUE, ORANGE, TEAL, GREY][i - 1], width=1.4),
                             showlegend=False), i, 1)
fig.update_layout(template="plotly_white", height=820, title=f"{DECOMPOSE_ITEM} · observed = trend × weekly × day-of-month × remainder")
fig.show()

In [ ]:
# 🎛️ Calendar covariates: catalogue, and the uplift each flag carries by group
catalogue = pd.DataFrame([{"feature": c, "family": "calendar (known in advance)", "description": COV_DOC.get(c, "")} for c in KNOWN]
                         + [{"feature": c, "family": "market driver (observed up to today)", "description": COV_DOC.get(c, "")} for c in PAST])
display(catalogue.style.set_caption("Covariates handed to the models").hide(axis="index"))

flags = [c for c in KNOWN if panel[c].dropna().isin([0, 1]).all()]
d = indexed(panel)
rows = {}
for c in flags:
    on = d[d[c] == 1].groupby("group")["idx"].mean()
    off = d[d[c] == 0].groupby("group")["idx"].mean()
    rows[c] = (on / off - 1) * 100
uplift = pd.DataFrame(rows).T
fig = px.imshow(uplift, color_continuous_scale="RdBu_r", zmin=-100, zmax=100, text_auto=".0f", aspect="auto",
                title="Average change in flow when a calendar flag is on (% vs days when it is off)")
fig.update_layout(template="plotly_white", height=40 * len(flags) + 160)
fig.show()

In [ ]:
# 🛢️ Market drivers: how flows respond to oil and rates
d = indexed(panel).merge(meta[["item_id", "product", "entity"]], on="item_id")
monthly = d.groupby(["product", pd.Grouper(key="timestamp", freq="MS")]).agg(idx=("idx", "mean"), brent=("brent_usd", "mean"),
                                                                             rate=("policy_rate", "mean")).reset_index()
fig = make_subplots(rows=1, cols=3, subplot_titles=("Trade finance vs Brent (monthly)", "Interbank funding vs policy rate (monthly)",
                                                     "Correlation of monthly flow with each driver"))
tf = monthly[monthly["product"] == "Trade finance settlements"]; ib = monthly[monthly["product"] == "Interbank funding"]
fig.add_trace(go.Scatter(x=tf.brent.shift(1), y=tf.idx, mode="markers", marker=dict(color=ORANGE, size=8), showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=ib.rate, y=ib.idx, mode="markers", marker=dict(color=TEAL, size=8), showlegend=False), 1, 2)
corr = monthly.groupby("product").apply(lambda g: pd.Series({"Brent (1-month lag)": g.idx.corr(g.brent.shift(1)), "policy rate": g.idx.corr(g.rate)}))
fig.add_trace(go.Heatmap(z=corr.values, x=corr.columns, y=corr.index, colorscale="RdBu_r", zmin=-1, zmax=1,
                         text=corr.round(2).values, texttemplate="%{text}", showscale=False), 1, 3)
fig.update_xaxes(title_text="Brent, previous month (USD)", row=1, col=1); fig.update_xaxes(title_text="policy rate (%)", row=1, col=2)
fig.update_layout(template="plotly_white", height=460, title="Why Brent and the policy rate are in the feature set")
fig.show()

In [ ]:
# 🧪 Which engineered features matter? A gradient-boosting probe on a time-ordered hold-out
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

feats = engineer_features(panel)
cut = feats.timestamp.quantile(0.85)
tr, te = feats[feats.timestamp <= cut], feats[feats.timestamp > cut]
cols = [c for c in feats.columns if c not in ("item_id", "timestamp", "y")]
fam = {c: feature_family(c) for c in cols}
print(f"{len(cols)} engineered features · {len(tr):,} training rows · {len(te):,} hold-out rows (after {cut:%d %b %Y})")
display(feats[["item_id", "timestamp", "y"] + cols[:8]].tail(3))

# Ablation: what each feature family adds
steps, used, maes = ["calendar", "lags", "rolling statistics", "market drivers"], [], {}
for s in steps:
    add = [c for c in cols if fam[c] == s]
    if not add:
        continue
    used += add
    m = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[used], tr["y"])
    maes["+ " + s if len(maes) else s] = mean_absolute_error(te["y"], m.predict(te[used]))
naive = mean_absolute_error(te["y"], te[f"lag_{SEASON}"])

probe = te.sample(min(6000, len(te)), random_state=0)
imp = permutation_importance(m, probe[used], probe["y"], n_repeats=3, random_state=0, scoring="neg_mean_absolute_error")
imp = pd.Series(imp.importances_mean, index=used).sort_values().tail(18)

fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6],
                    subplot_titles=("Hold-out error as feature families are added", "Permutation importance (top 18)"))
fig.add_trace(go.Bar(x=["seasonal naive"] + list(maes), y=[naive] + list(maes.values()),
                     marker_color=[GREY] + [FAMILY_COLORS[s.replace("+ ", "")] for s in maes],
                     text=[f"{v:.3f}" for v in [naive] + list(maes.values())], textposition="outside", showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=imp.values, y=imp.index, orientation="h", marker_color=[FAMILY_COLORS[fam[c]] for c in imp.index],
                     showlegend=False), 1, 2)
for f_, c_ in FAMILY_COLORS.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=c_, name=f_), 1, 2)
fig.update_yaxes(title_text="MAE (indexed units)", row=1, col=1)
fig.update_layout(template="plotly_white", height=560, title="Feature engineering pays: lags, rolling statistics and calendar",
                  legend=dict(orientation="h", y=-0.1))
fig.show()

## 4 · Model race on rolling backtests

`TimeSeriesLeaderboard` scores every model on the last three 4-week windows of history, never on data it was fitted on. Foundation models run **zero-shot**. **MASE** below 1 beats a seasonal-naive forecast; **WQL** scores the whole predictive distribution.

In [ ]:
# 🏁 Hold out the final horizon, race the models on rolling backtests of the rest
_past = {"past_covariates": PAST} if PAST else {}
schema = TimeSeriesSchema(target="y", item_id="item_id")
schema_cov = TimeSeriesSchema(target="y", item_id="item_id", known_covariates=KNOWN, **_past)
model_df = panel[["item_id", "timestamp", "y", *KNOWN, *PAST]]
history, actual = split_horizon(model_df, schema_cov, prediction_length=HORIZON)
future_cov = actual[["item_id", "timestamp", *KNOWN]]
history_y = history[["item_id", "timestamp", "y"]]
value_hist = panel.merge(history[["item_id", "timestamp"]], on=["item_id", "timestamp"])
value_act = panel.merge(actual[["item_id", "timestamp"]], on=["item_id", "timestamp"])
fp = {"prediction_length": HORIZON, "quantile_levels": QUANTILES}

board = TimeSeriesLeaderboard(history_y, schema, forecast_params=fp, windows=3)
board.add_models(BASELINES + FOUNDATION_MODELS)
t0 = time.time()
ranking = board.run(rank_by="mase", display=False)
print(f"{history.item_id.nunique()} series · {len(history):,} history rows · hold-out {HORIZON} steps · leaderboard in {time.time() - t0:.0f}s")
display(ranking)

ok = ranking[ranking["Status"].eq("ok")] if "Status" in ranking else ranking
fig = make_subplots(rows=1, cols=2, subplot_titles=("MASE (lower is better)", "WQL (lower is better)"))
for i, metric in enumerate(["mase", "wql"], start=1):
    o = ok.sort_values(metric)
    fig.add_trace(go.Bar(y=o["Model"], x=o[metric], orientation="h", text=o[metric].round(3), textposition="outside",
                         marker_color=[BLUE if m in FOUNDATION_MODELS else GREY for m in o["Model"]], showlegend=False), 1, i)
fig.add_vline(x=1.0, line_dash="dot", line_color=DARK, row=1, col=1)
fig.update_layout(template="plotly_white", height=420,
                  title="Rolling-backtest leaderboard · blue = foundation model (zero-shot), grey = baseline")
fig.show()

## 5 · Four-week forecast with covariates and calibrated bands

In [ ]:
# 🔮 Forecast with and without covariates, then calibrate the intervals
plain = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial", tuning_params={"device": DEVICE})
plain.fit(history_y, schema)
m_plain = plain.evaluate(actual)

pipe = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial", tuning_params={"device": DEVICE})
pipe.fit(history, schema_cov, future_df=future_cov)
m_cov = pipe.evaluate(actual, forecast=pipe.predict())

pipe.calibrate(windows=4)
forecast_cal = pipe.predict()
m_cal = pipe.evaluate(actual, forecast=forecast_cal)

KEYS = [k for k in ("mase", "wql", "smape", "coverage_80", "coverage_90") if k in m_cov]
display(pd.DataFrame({"target only": m_plain, "with covariates": m_cov, "with covariates, calibrated": m_cal}).loc[KEYS])
gain = (1 - m_cov["mase"] / m_plain["mase"]) * 100
display(Markdown(f"**Covariates change MASE by {gain:+.1f}%** on the hold-out ({PRIMARY})."))

fc = forecast_cal.to_pandas()
QCOLS = ["point"] + [str(q) for q in QUANTILES]
fc[QCOLS] = fc[QCOLS].mul(fc.item_id.map(SCALE), axis=0)          # back to business units

In [ ]:
# 🌀 Interactive fan chart: pick a series from the dropdown
tail = value_hist[value_hist.timestamp > value_hist.timestamp.max() - pd.Timedelta(days=150)]
items = list(meta.item_id)
fig = go.Figure()
for k, item in enumerate(items):
    h = tail[tail.item_id == item]; a = value_act[value_act.item_id == item]; f = fc[fc.item_id == item]
    vis = k == 0
    fig.add_trace(go.Scatter(x=h.timestamp, y=h.value, name="history", line=dict(color=DARK, width=1.5), visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.95"], f["0.05"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.12)", line=dict(width=0), name="90% band", visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.9"], f["0.1"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.25)", line=dict(width=0), name="80% band", visible=vis))
    fig.add_trace(go.Scatter(x=f.timestamp, y=f["0.5"], name="median forecast", line=dict(color=BLUE, width=2.5), visible=vis))
    fig.add_trace(go.Scatter(x=a.timestamp, y=a.value, name="actual", mode="markers", marker=dict(color=ORANGE, size=6), visible=vis))
n = 5
buttons = [dict(label=item, method="update",
                args=[{"visible": [j // n == k for j in range(n * len(items))]}, {"title": f"{item} · forecast"}])
           for k, item in enumerate(items)]
fig.update_layout(template="plotly_white", height=520, title=f"{items[0]} · forecast", yaxis_title=f"{UNIT} per day",
                  updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.15, direction="down")],
                  legend=dict(orientation="h", y=-0.15))
fig.show()

# Accuracy by series: where the model is strong and where it is not
per = fc.merge(value_act[["item_id", "timestamp", "value"]], on=["item_id", "timestamp"])
acc = per.groupby("item_id").apply(lambda g: pd.Series({
    "wape_%": (g["0.5"] - g["value"]).abs().sum() / max(g["value"].abs().sum(), 1e-9) * 100,
    "inside_90_%": g["value"].between(g["0.05"], g["0.95"]).mean() * 100})).reset_index()
acc["group"] = acc.item_id.map(GROUP_OF)
fig = px.bar(acc.sort_values("wape_%"), x="item_id", y="wape_%", color="group", color_discrete_map=GROUP_COLORS,
             title="Hold-out error by series (weighted absolute % error of the median forecast)")
fig.update_layout(template="plotly_white", height=430, xaxis_tickangle=-60, xaxis_title="")
fig.show()

## 6 · Group liquidity dashboard

In [ ]:
# 🧭 Roll the series forecasts up the hierarchy: entity × business line
fcs = fc.merge(meta, on="item_id")
need = fcs.groupby("group")[["0.5", "0.95"]].sum()
per = fc.merge(value_act[["item_id", "timestamp", "value"]], on=["item_id", "timestamp"])
grid_ = fcs.groupby(["entity", "group"])["0.5"].sum().unstack()

fig = make_subplots(rows=2, cols=4, row_heights=[0.3, 0.7], vertical_spacing=0.13,
                    specs=[[{"type": "indicator"}] * 4, [{"colspan": 2, "secondary_y": True}, None, {"colspan": 2}, None]],
                    subplot_titles=("", "", "", "", "Daily outlook by business line (median, USD mn)",
                                    "4-week requirement by entity and line (median, USD mn)"))
kpis = [("4-week flow<br>median, USD mn", need["0.5"].sum()), ("Stress buffer<br>to P95, USD mn", (need["0.95"] - need["0.5"]).sum()),
        ("Forecast error<br>hold-out MASE", m_cal["mase"]), ("Outcomes inside<br>90% band", per.value.between(per["0.05"], per["0.95"]).mean() * 100)]
for i, (title, v) in enumerate(kpis, 1):
    fig.add_trace(go.Indicator(mode="number", value=v, title={"text": title},
                               number={"valueformat": ",.2f" if v < 50 else ",.0f", "suffix": "%" if "band" in title else "",
                                       "font": {"size": 46}}), 1, i)
daily = fcs.groupby(["group", "timestamp"])["0.5"].sum().reset_index()
for gname, g in daily.groupby("group"):
    fig.add_trace(go.Scatter(x=g.timestamp, y=g["0.5"], name=gname + (" (right axis)" if gname == "Retail" else ""),
                             line=dict(color=GROUP_COLORS[gname], width=2.5)), 2, 1, secondary_y=gname == "Retail")
for gname in grid_.columns:
    fig.add_trace(go.Bar(x=grid_.index, y=grid_[gname], marker_color=GROUP_COLORS[gname], showlegend=False,
                         text=grid_[gname], texttemplate="%{text:,.0f}"), 2, 3)
fig.update_layout(template="plotly_white", height=740, barmode="stack", legend=dict(orientation="h", y=-0.08),
                  title=f"Group liquidity outlook · {fc.timestamp.min():%d %b} – {fc.timestamp.max():%d %b %Y} · model {PRIMARY}")
fig.show()

tot = per.merge(meta, on="item_id").groupby(["entity", "group"]).agg(forecast=("0.5", "sum"), actual=("value", "sum"))
tot["error_%"] = (tot.forecast / tot.actual - 1) * 100
display(tot.style.format({"forecast": "{:,.0f}", "actual": "{:,.0f}", "error_%": "{:+.1f}%"})
        .background_gradient(subset=["error_%"], cmap="RdBu_r", vmin=-25, vmax=25)
        .set_caption("4-week totals: forecast vs what actually happened (hold-out)"))

## 7 · Ensembles and series similarity

In [ ]:
# 🤝 Greedy ensemble, and series similarity from foundation-model embeddings
ensemble = TimeSeriesEnsemble(BASELINES + FOUNDATION_MODELS, ensemble_strategy="greedy_selection",
                              forecast_params=fp, verbose=False)
ensemble.fit(history_y, schema)
display(ensemble.get_leaderboard())
m_ens = ensemble.evaluate(actual)
display(pd.Series({k: m_ens[k] for k in ("mase", "wql") if k in m_ens}, name="ensemble hold-out").to_frame())

if EMBEDDER:
    emb = TimeSeriesPipeline(EMBEDDER, task_type="embedding", tuning_params={"device": DEVICE}).fit(history_y, schema).predict()
    sim = pd.DataFrame(emb.similarity(), index=list(emb.item_ids), columns=list(emb.item_ids))
    order = meta.sort_values("group").item_id
    fig = px.imshow(sim.loc[order, order], color_continuous_scale="Blues", zmax=1,
                    title=f"Cosine similarity of {EMBEDDER} embeddings · {emb.dim}-dim per series")
    fig.update_layout(height=720, template="plotly_white"); fig.show()
else:
    print("Embeddings need a foundation model (skipped in fast mode).")

if RUN_FINETUNE and not FAST:
    tuned = TimeSeriesPipeline("ChronosBolt", tuning_strategy="peft", forecast_params=fp,
                               tuning_params={"epochs": 2, "steps_per_epoch": 50, "device": DEVICE}).fit(history_y, schema)
    zero = TimeSeriesPipeline("ChronosBolt", forecast_params=fp, tuning_params={"device": DEVICE}).fit(history_y, schema)
    display(pd.DataFrame({"zero-shot": zero.evaluate(actual), "LoRA, 2 epochs": tuned.evaluate(actual)}).loc[["mase", "wql"]])

## 8 · Anomaly detection on a live feed

The last 480 days become a **live monitoring feed**; labelled incidents are injected into its final 150 days:

| Incident | Looks like | Typical cause |
|---|---|---|
| Spike | one day at 3–5× normal | suspected fraud, large one-off |
| Outage | 1–3 days near zero | channel outage, failed feed |
| Level shift | 5–10 days at ±40–60% | migration, reporting break |

Each day is scored against rolling forecasts made **only from earlier history**, turned into a conformal p-value against a clean reference period, and flagged below the false-alarm rate `alpha`.

In [ ]:
def monitor_figure(res, feed_df, items, title_suffix, truth=None, events=None):
    """Expected band, observed flow, alerts (and true incidents or known events) with a series dropdown."""
    fig = go.Figure()
    per = 5
    for k, item in enumerate(items):
        r = res[res.item_id == item]; f = feed_df[feed_df.item_id == item]; vis = k == 0
        fig.add_trace(go.Scatter(x=pd.concat([r.timestamp, r.timestamp[::-1]]), y=pd.concat([r.upper, r.lower[::-1]]),
                                 fill="toself", fillcolor="rgba(47,91,234,0.14)", line=dict(width=0), name="expected range", visible=vis))
        fig.add_trace(go.Scatter(x=f.timestamp, y=f.y, name="observed", line=dict(color=DARK, width=1.3), visible=vis))
        if truth is not None:
            t = r.merge(truth[truth.is_anomaly], on=["item_id", "timestamp"])
            fig.add_trace(go.Scatter(x=t.timestamp, y=t.value, mode="markers", name="true incident", visible=vis,
                                     marker=dict(symbol="circle-open", size=14, color=ORANGE, line=dict(width=2))))
        else:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="", showlegend=False, visible=vis))
        a = r[r.is_anomaly.fillna(False).astype(bool)]
        fig.add_trace(go.Scatter(x=a.timestamp, y=a.value, mode="markers", name="TabTune alert", visible=vis,
                                 marker=dict(symbol="x", size=9, color=RED)))
        fig.add_trace(go.Scatter(x=r.timestamp, y=r.score, name="anomaly score", yaxis="y2", line=dict(color=GREY, width=1), visible=vis))
    buttons = [dict(label=item, method="update",
                    args=[{"visible": [j // per == k for j in range(per * len(items))]}, {"title": f"{item} · {title_suffix}"}])
               for k, item in enumerate(items)]
    for when, label in (events or []):
        fig.add_vline(x=pd.Timestamp(when), line_dash="dot", line_color=GOLD)
        fig.add_annotation(x=pd.Timestamp(when), y=1.02, yref="paper", text=label, showarrow=False, textangle=-20, font=dict(size=10))
    fig.update_layout(template="plotly_white", height=580, title=f"{items[0]} · {title_suffix}",
                      yaxis=dict(title="indexed flow", domain=[0.3, 1]), yaxis2=dict(title="score", domain=[0, 0.22]),
                      updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.18)], legend=dict(orientation="h", y=-0.12))
    return fig

In [ ]:
# 🚨 Build the feed, inject labelled incidents, run three detectors
rng = np.random.default_rng(11)
LIVE_DAYS, INJECT_DAYS = 480, 150
live = panel[panel.timestamp > panel.timestamp.max() - pd.Timedelta(days=LIVE_DAYS)][["item_id", "timestamp", "y"]].copy()
live = live.sort_values(["item_id", "timestamp"]).reset_index(drop=True)
live["is_anomaly"] = False
events, cutoff = [], live.timestamp.max() - pd.Timedelta(days=INJECT_DAYS)
for item, idx in live.groupby("item_id").indices.items():
    idx = np.asarray(idx); window = idx[live.loc[idx, "timestamp"].values > cutoff]; used = set()
    for kind in rng.choice(["spike", "spike", "outage", "level_shift"], size=rng.integers(3, 6)):
        length = {"spike": 1, "outage": int(rng.integers(1, 4)), "level_shift": int(rng.integers(5, 11))}[kind]
        for _ in range(20):
            start = int(rng.integers(0, len(window) - length)); span = set(range(start, start + length))
            if not span & used:
                break
        used |= span; rows = window[start:start + length]
        factor = {"spike": rng.uniform(3, 5), "outage": rng.uniform(0.02, 0.15),
                  "level_shift": rng.choice([rng.uniform(1.4, 1.6), rng.uniform(0.4, 0.6)])}[kind]
        live.loc[rows, "y"] *= factor; live.loc[rows, "is_anomaly"] = True
        events.append({"item_id": item, "start": live.loc[rows[0], "timestamp"], "days": length, "type": kind})
events = pd.DataFrame(events)
labels, feed = live[["item_id", "timestamp", "is_anomaly"]], live.drop(columns="is_anomaly")
print(f"{len(events)} incidents · {int(labels.is_anomaly.sum())} anomalous days out of {len(live):,}")

from sklearn.metrics import roc_auc_score, average_precision_score
TASK = {"min_context": 240, "alpha": 0.02, "reference_fraction": 0.3}
results, scores = {}, []
for method in ("interval", "forecast_error", "likelihood"):
    t0 = time.time()
    det = TimeSeriesPipeline(DETECTOR, task_type="anomaly_detection", tuning_params={"device": DEVICE},
                             task_params={"method": method, **TASK})
    results[method] = det.fit(feed, schema).predict()
    scores.append({"method": method, **results[method].evaluate(labels), "seconds": round(time.time() - t0, 1)})
scores = pd.DataFrame(scores).set_index("method")

def rolling_median_score(v):                                   # classic robust baseline
    med = v.rolling(29, center=True, min_periods=7).median()
    return (v - med).abs() / (1.4826 * (v - med).abs().rolling(29, center=True, min_periods=7).median() + 1e-9)
scored = results["interval"].to_pandas()[["item_id", "timestamp", "score"]].dropna()
chk = (scored.merge(feed.assign(base=feed.groupby("item_id")["y"].transform(rolling_median_score)), on=["item_id", "timestamp"])
       .merge(labels, on=["item_id", "timestamp"]))
scores.loc["rolling median (baseline)", ["auroc", "auprc"]] = [roc_auc_score(chk.is_anomaly, chk.base), average_precision_score(chk.is_anomaly, chk.base)]
display(scores[["n", "n_anomalies", "auroc", "auprc", "precision", "recall", "f1", "seconds"]])
BEST = scores.drop("rolling median (baseline)")["auprc"].idxmax()
display(Markdown(f"**Best detector by AUPRC: `{BEST}`** with {DETECTOR}. Always read it next to the simple baseline."))

In [ ]:
# 🖥️ Monitoring dashboard and alert analytics
res = results[BEST].to_pandas()
monitor_figure(res, feed, list(meta.item_id), "live monitoring", truth=labels).show()

alerts = results[BEST].anomalies.merge(meta, on="item_id").merge(labels.rename(columns={"is_anomaly": "confirmed_incident"}), on=["item_id", "timestamp"])
alerts["deviation_%"] = (alerts.value / alerts.expected.replace(0, np.nan) - 1) * 100
queue = alerts.sort_values("score", ascending=False)[["timestamp", "item_id", "group", "value", "expected", "deviation_%", "p_value", "confirmed_incident"]].head(15)
display(queue.style.format({"value": "{:,.2f}", "expected": "{:,.2f}", "deviation_%": "{:+.0f}%", "p_value": "{:.4f}"})
        .background_gradient(subset=["deviation_%"], cmap="RdBu_r", vmin=-100, vmax=100).set_caption("Top 15 alerts by anomaly score"))

flagged = set(zip(alerts.item_id, alerts.timestamp))
events["detected"] = [any((e.item_id, d) in flagged for d in pd.date_range(e.start, periods=e.days)) for e in events.itertuples()]
by_type = events.groupby("type")["detected"].mean().mul(100).reset_index()
weeks = pd.date_range(feed.timestamp.max() - pd.Timedelta(days=INJECT_DAYS + 60), feed.timestamp.max(), freq="W")
weekly = (alerts.groupby(["item_id", pd.Grouper(key="timestamp", freq="W")]).size().unstack(fill_value=0)
          .reindex(index=list(meta.sort_values("group").item_id), columns=weeks, fill_value=0))
fig = make_subplots(rows=1, cols=2, column_widths=[0.3, 0.7], subplot_titles=("Incidents detected by type (%)", "Alerts per series per week"))
fig.add_trace(go.Bar(x=by_type.type, y=by_type.detected, marker_color=[BLUE, ORANGE, TEAL][:len(by_type)],
                     text=by_type.detected.round(0), textposition="outside", showlegend=False), 1, 1)
fig.add_trace(go.Heatmap(z=weekly.values, x=weekly.columns, y=weekly.index, colorscale=[[0, "#F7F7F4"], [1, ORANGE]], zmin=0,
                         colorbar=dict(title="alerts")), 1, 2)
fig.update_yaxes(range=[0, 115], row=1, col=1)
fig.update_layout(template="plotly_white", height=760, title=f"Alert analytics · {DETECTOR} · method {BEST}")
fig.show()
print(f"incidents detected: {int(events.detected.sum())} of {len(events)} · alerts raised: {len(alerts)} · "
      f"alerts on confirmed incident days: {alerts.confirmed_incident.mean():.0%}")

## Summary

| Stage | What was shown | TabTune call |
|---|---|---|
| Data preparation | de-duplication, null handling, label harmonisation, unit-error repair, value-date aggregation, FX conversion, outage imputation — validated against ground truth | — |
| Feature engineering | scaling, seasonality strength, autocorrelation, decomposition, calendar uplift, market drivers, feature-family ablation | `TimeSeriesSchema(known_covariates=, past_covariates=)` |
| Model selection | rolling-backtest leaderboard, foundation models vs baselines | `TimeSeriesLeaderboard` |
| Forecasting | covariates, conformal calibration, fan charts | `TimeSeriesPipeline.fit / predict / calibrate` |
| Roll-up | entity × business-line liquidity dashboard | — |
| Ensembles, embeddings | greedy ensemble, similarity map | `TimeSeriesEnsemble`, `task_type="embedding"` |
| Monitoring | three detectors scored against labelled incidents and a simple baseline | `task_type="anomaly_detection"` |

To run on a bank's own data, replace section 1 with the real ledger extract in the same ten-column layout; every later cell runs unchanged.